# S09 - Assessment Test 3

## Libraries

In [1]:
import os
import sqlite3 as sql
import pandas as pd

## Functions

In [2]:
# Function taking a SQL query file to format its text for Jupyter Notebook
def query_text_format(path, file):
    # Change directory
    try:
        os.chdir(path)
    except FileNotFoundError:
        print(f"El directorio {path} no se encontró.")
        return
    
    # Read file
    try:
        with open(file, 'r') as f:
            query_log = f.read()
    except FileNotFoundError:
        print(f"El archivo {file} no se encontró.")
        return
    
    # Text formatting
    # Removal of SQL comment closing
    query_log = query_log.replace(' */', '')
    # Split into lines
    query_log_lines = query_log.split('\n')
    
    # Enumerated iteration over all file lines
    for i, line in enumerate(query_log_lines):
        query_log_lines[i] = line.strip()
        
        # Replacement of SQL comment opening
        if query_log_lines[i][:2] == '/*':
            # Search two integers to detect lesson start
            double_int = []            
            for char in line[3:5]:
                try:
                    if isinstance(int(char), int):
                        double_int.append(1)
                except:
                    pass
            
            # If two integers found, replacement according to lesson start
            if len(double_int) == 2:
                query_log_lines[i] = query_log_lines[i].replace('/*', '###')
            # If a letter, replacement according to assessment exercise start
            elif query_log_lines[i][3].isalpha() and query_log_lines[i][4] == '.':
                query_log_lines[i] = query_log_lines[i].replace('/*', '####')
            # Remaining SQL comments are Python comments, too
            else:
                query_log_lines[i] = query_log_lines[i].replace('/*', '#')
        
        # If line not empty
        elif line != '':
            query_log_lines[i] = '\t' + query_log_lines[i] + ' \\'
        
        # Rest of cases (potential)
        else:
            pass
        
    return '\n'.join(query_log_lines)

In [3]:
# Function taking the SQL queries and building Python code to execute them
def build_python_queries(text):
    # Split into lines
    query_lines = text.split('\n')
    
    # Build SQL queries in Python
    resulting_text = []
    flg_first_sql_line = True
    
    for i, line in enumerate(query_lines):
        # Empty line
        if line == '':
            resulting_text.append(line)
        # Line starting with tab
        elif query_lines[i][0] == '\t':
            # First query line
            if flg_first_sql_line:
                # First query line in one line query
                if (i + 1 <= len(query_lines) - 1) and ((query_lines[i + 1] == '') or ((query_lines[i + 1][0] != '') and (query_lines[i + 1][0] == '#'))):
                    resulting_text.append('query = " \\')
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                # First query line in multiple lines query
                else:
                    resulting_text.append('query = " \\')
                    resulting_text.append(line)
                    flg_first_sql_line = False
            # Second, or other, query line
            else:
                # Last query line
                if (i + 1 <= len(query_lines) - 1) and ((query_lines[i + 1] == '') or ((query_lines[i + 1][0] != '') and (query_lines[i + 1][0] == '#'))):
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                    flg_first_sql_line = True
                # Query line, absolute last one
                elif i == len(query_lines) - 1:
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                # Query lines that are not first nor last
                else:
                    resulting_text.append(line)
        # Other lines
        else:
            resulting_text.append(line)
    
    return '\n'.join(resulting_text)

In [4]:
# Function taking the SQL query text and executing it
def execute_query(query_text, connection):
    try:
        # Check if the query is a SELECT statement
        if query_text.strip().upper().startswith("SELECT"):
            # DataFrame from query
            results_df = pd.read_sql_query(
                query_text,
                connection
            )
            print(results_df)
        else:
            # Execute queries that do not return results
            with connection:
                connection.execute(query_text)
            print("Query executed successfully.")
    except Exception as e:
        print(f"Error executing the query: {e}")

## Settings

In [5]:
# Limit removal for showing pandas.DataFrames' columns
pd.set_option('display.max_columns', None)
# Limit removal for showing pandas.DataFrames' rows
pd.set_option('display.max_rows', None)
# Modification of console with for displaying
pd.set_option('display.width', 8000)

## SQL Query Formatting

In [6]:
# First formatting
path = 'G:\\15_Estudio\\Udemy\\SQL - Portilla Complete Bootcamp'
file = 'UDM-SQL-BTCMP--009-Assessment.sql'

first_pass = query_text_format(path, file)
print(first_pass)

### 70. Assessment Test 3

	Welcome to your final assessment test! This will test your knowledge of the \
	previous section, focused on creating databases and table operations. \
	This test will actually consist of a more open-ended assignment below: \

	Complete the following task: \
	Create a new database called "School" this database should have two tables: \
	teachers and students. \

	The students table should have columns for student_id, first_name, last_name, \
	homeroom_number, phone,email, and graduation year. \

	The teachers table should have columns for teacher_id, first_name, last_name, \
	homeroom_number, department, email, and phone. \

	The constraints are mostly up to you, but your table constraints \
	do have to consider the following: \
	- We must have a phone number to contact students in case of an emergency. \
	- We must have ids as the primary key of the tables. \
	- Phone numbers and emails must be unique to the individual. \

	Once you've made the tables, inser

In [7]:
# Final formatting
working_text = build_python_queries(first_pass)
print(working_text)

### 70. Assessment Test 3

query = " \
	Welcome to your final assessment test! This will test your knowledge of the \
	previous section, focused on creating databases and table operations. \
	This test will actually consist of a more open-ended assignment below: \
	"
execute_query(query, conn)

query = " \
	Complete the following task: \
	Create a new database called "School" this database should have two tables: \
	teachers and students. \
	"
execute_query(query, conn)

query = " \
	The students table should have columns for student_id, first_name, last_name, \
	homeroom_number, phone,email, and graduation year. \
	"
execute_query(query, conn)

query = " \
	The teachers table should have columns for teacher_id, first_name, last_name, \
	homeroom_number, department, email, and phone. \
	"
execute_query(query, conn)

query = " \
	The constraints are mostly up to you, but your table constraints \
	do have to consider the following: \
	- We must have a phone number to contact students in 

This text will be used to create the whole of the Practice section in this notebook.

## Assessment

### 70. Assessment Test 3

Welcome to your final assessment test! This will test your knowledge of the previous section, focused on creating databases and table operations. This test will actually consist of a more open-ended assignment below:

**Complete the following task:**

- Create a new database called "School" this database should have two tables:
    - teachers and students.

- The students table should have columns for student_id, first_name, last_name, homeroom_number, phone,email, and graduation year.

- The teachers table should have columns for teacher_id, first_name, last_name, homeroom_number, department, email, and phone.

- The constraints are mostly up to you, but your table constraints do have to consider the following:
    - We must have a phone number to contact students in case of an emergency.
    - We must have ids as the primary key of the tables.
    - Phone numbers and emails must be unique to the individual.

- Once you've made the tables, insert a student named Mark Watney (student_id=1) who has a phone number of 777-555-1234 and doesn't have an email. He graduates in 2035 and has 5 as a homeroom number.

- Then insert a teacher names Jonas Salk (teacher_id = 1) who as a homeroom number of 5 and is from the Biology department. His contact info is: jsalk@school.org and a phone number of 777-555-4321.

- Keep in mind that these insert tasks may affect your constraints.

### Connection with Data Base

In [8]:
# Connection and cursor
conn = sql.connect('G:\\15_Estudio\\Udemy\\SQL - Portilla Complete Bootcamp\\School.db')
cur = conn.cursor()

### SOLUTION

In [9]:
# Table creation: teachers, students
query = " \
	CREATE TABLE teachers( \
	teacher_id SERIAL PRIMARY KEY, \
	first_name VARCHAR(100) NOT NULL, \
	last_name VARCHAR(100) NOT NULL, \
	homeroom_number INTEGER NOT NULL, \
	department VARCHAR(50) NOT NULL, \
	phone VARCHAR(50) UNIQUE, \
	email VARCHAR(100) UNIQUE \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [10]:
query = " \
	CREATE TABLE students( \
	student_id SERIAL PRIMARY KEY, \
	first_name VARCHAR(100) NOT NULL, \
	last_name VARCHAR(100) NOT NULL, \
	homeroom_number INTEGER NOT NULL, \
	phone VARCHAR(50) NOT NULL UNIQUE, \
	email VARCHAR(100) UNIQUE, \
	grad_year INTEGER \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [11]:
# Student register insertion
query = " \
	INSERT INTO students( \
	first_name, \
	last_name, \
	homeroom_number, \
	phone, \
	grad_year \
	) \
	VALUES( \
	'Mark', \
	'Watney', \
	5, \
	'777-555-1234', \
	2035 \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [12]:
# Verification
query = " \
	SELECT * \
	FROM students; \
	"
execute_query(query, conn)

  student_id first_name last_name  homeroom_number         phone email  grad_year
0       None       Mark    Watney                5  777-555-1234  None       2035


In [13]:
# Teacher register insertion
query = " \
	INSERT INTO teachers( \
	first_name, \
	last_name, \
	homeroom_number, \
	department, \
	phone, \
	email \
	) \
	VALUES( \
	'Jonas', \
	'Salk', \
	5, \
	'Biology', \
	'777-555-4321', \
	'jsalk@school.org' \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [14]:
# Verification
query = " \
	SELECT * \
	FROM teachers; \
	"
execute_query(query, conn)

  teacher_id first_name last_name  homeroom_number department         phone             email
0       None      Jonas      Salk                5    Biology  777-555-4321  jsalk@school.org
